# 10. Turnover: membership, reweighting and rebalancing, and E7

**What this notebook does.** It splits each path's monthly *turnover*, the share of the portfolio sold and bought, into three parts, each measured, sums them by year, and judges E7.

**Terms used in this notebook.**

- *one-way turnover*: half the sum of the absolute changes in weight between two portfolios. Every turnover in this notebook is one-way.
- *drifted portfolio*: last month's portfolio after last month's returns. The *total turnover* of a month runs from it to the new weights.
- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *benchmark*: the universe weighted by market capitalisation at the start of the month.
- *reference portfolio*: the drifted portfolio with the stocks that left the universe sold, the entrants bought at their benchmark weight, and the others rescaled to sum to one.
- *membership*, the *forced turnover*: the turnover from the drifted portfolio to the reference portfolio.
- *restored portfolio*: each continuing stock at last month's chosen weight times the ratio of its benchmark weights this month and last, every entrant at its benchmark weight, rescaled to one. It keeps each stock's weight relative to its benchmark weight where the optimiser left it.
- *reweighting*: the turnover from the reference portfolio to the restored portfolio, which following the benchmark's own reweighting takes. It comes from changes in share counts, and from dividends, because the drifted portfolio grows by total returns and the benchmark's weights by prices.
- *rebalancing*: the turnover from the restored portfolio to the new weights, the optimiser's own moves. It is zero for a path that holds the benchmark.
- *overlap*: the three parts less the total, the trades of one step that the next step undoes. By the triangle inequality it is never below zero.
- *chosen turnover*: the turnover from the reference portfolio to the new weights, the quantity the cap bounds.
- *mandate*: constraint set C3, the portfolio the study runs, set out in notebook 08.
- *verdict*: the record of whether an expectation fixed before the run is met.

**The benchmark's own figures.** Counting the change from June to July 1999, the benchmark alone turns over 5.82 per cent a year over the 306 months to December 2024: membership 2.84, reweighting 3.73 and an overlap of 0.75. A path that holds the benchmark starts in July 1999 with no turnover in that month, so section 2 shows it at 5.78, 2.80, 3.71 and 0.73. Restoring last month's weights themselves undoes the month's price moves, and on the benchmark alone it would take 30.64 per cent a year, trading that no path does. A middle part defined that way would leave the rest below zero in every month, so the middle part follows the benchmark's own weights. These figures were computed on 12 November 2026 at 14:46, from this study's panel by code that is not in this repository, before any path's turnover was read. They describe the benchmark alone and say nothing of the mandate.

**What I expect to see, fixed in `src/slt/constants.py` before any path ran.**

- *E7*: under the mandate at the 2 per cent cap on the Ledoit-Wolf estimator, membership is less than a third of total turnover over the window. The index replaces about four to five per cent of its weight a year, and a cap of 2 per cent a month allows about 24 per cent a year of chosen turnover.
- With no verdict: under C0 and C1, where the portfolio is the benchmark, rebalancing is zero to the solver's precision, and the other figures are close to the benchmark's own.

Running time: under a minute.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
import numpy as np

from slt import evaluation as V

OUT = ROOT / C.OUTPUTS_DIR / "evaluation"

## 1. The decomposition, per path, over the window and by year

Per year over the window, in per cent: the total, the three parts and the overlap. The table by year is committed in section 3.

In [ ]:
records = V.load_records(OUT)
figures = [f"turnover_{k}" for k in C.TURNOVER_FIGURES]
table = pd.concat([V.turnover_by_year(rec).assign(path=lab) for lab, rec in records.items()], ignore_index=True)
table = table[["path"] + [c for c in table.columns if c != "path"]]
years = pd.Series({lab: len(rec) / 12 for lab, rec in records.items()})
window = table.groupby("path", sort=False)[figures].sum().div(years, axis=0)
print((100 * window).round(2).to_string())
low = min(float(rec["turnover_overlap"].min()) for rec in records.values())
print(f"smallest overlap in any month of any path: {low:.2e}")

## 2. The paths that hold the benchmark, beside the benchmark's own figures

In [ ]:
own = {"turnover_total": 5.82, "turnover_membership": 2.84, "turnover_reweighting": 3.73, "turnover_overlap": 0.75}
for lab in (V.label("sample", "C0", None), V.label("sample", "C1", None)):
    row = 100 * window.loc[lab]
    print(lab + ": " + " | ".join(f"{k[9:]} {row[k]:.2f}" + (f" (benchmark {own[k]:.2f})" if k in own else "") for k in figures))

## 3. E7

In [ ]:
summaries = {lab: V.summarise(rec) for lab, rec in records.items()}
e7 = V.judge(summaries)["E7"]
print(f"E7: {'met' if e7['met'] else 'not met'}. Under the mandate at the 2 per cent cap on the Ledoit-Wolf estimator, membership is "
      f"{100 * e7['membership_share_of_total']:.1f} per cent of total one-way turnover over the window, against less than {100 * C.E7_FORCED_SHARE_MAX:.1f}.")

## 4. What is committed

Two files: the decomposition by path and year, the total, the three parts and the overlap, and the verdicts of milestone 3 with E7 added.

In [ ]:
path = ROOT / C.RESULTS_DIR / "verdicts_milestone3.json"
record = json.loads(path.read_text()) if path.exists() else {}
record["E7"] = V.jsonable(e7)
for p in [X.write_results("turnover_by_year.csv", frame=table), X.write_results("verdicts_milestone3.json", record=record)]:
    print("written:", p.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: how much of each path's trading the index's own changes force, how much following the benchmark's own reweighting takes, how much the optimiser chooses, and whether the forced part is the small one E7 expects.

What could still be wrong. The parts depend on the order of the steps, membership first, then reweighting, then rebalancing: another order would split the same total another way, and the overlap says how much the steps undo of each other. And the forced part counts only the trades the index's changes require on the day the month begins. A portfolio that anticipated a change would trade earlier, and this study does not.